In [ ]:
!pip -q install transformers datasets accelerate evaluate \
    scikit-learn gradio pymupdf

In [ ]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Enable GPU in Runtime settings.")

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
GPU available: True
GPU: Tesla T4


In [ ]:

from datasets import load_dataset

# Load SQuAD 2.0 from Hugging Face
dataset = load_dataset("rajpurkar/squad_v2")

print(dataset)
print("\nTraining examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))

# Inspect one example
sample = dataset["train"][0]

print("\nQuestion:", sample["question"])
print("\nContext:", sample["context"][:300])
print("\nAnswer:", sample["answers"])


README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

Training examples: 130319
Validation examples: 11873

Question: When did Beyonce start becoming popular?

Context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singe

Answer: {'text': ['in the late 1990s'], 'answer_start': [269]}


In [ ]:

# Create manageable subsets for the first experiment
train_data = dataset["train"].shuffle(seed=42).select(range(10000))

validation_data = (
    dataset["validation"]
    .shuffle(seed=42)
    .select(range(1000))
)

print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))


Training examples: 10000
Validation examples: 1000


In [ ]:

from transformers import AutoTokenizer

MODEL_NAME = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Tokenizer loaded successfully")
print("Model:", MODEL_NAME)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer loaded successfully
Model: FacebookAI/roberta-base


In [ ]:

MAX_LENGTH = 384
DOC_STRIDE = 128

def preprocess_qa(examples):
    questions = [q.strip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offsets_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offsets_mapping):
        sample_idx = sample_mapping[i]
        input_ids = tokenized["input_ids"][i]
        sequence_ids = tokenized.sequence_ids(i)

        cls_index = input_ids.index(tokenizer.cls_token_id)
        answer = examples["answers"][sample_idx]

        # Default: no answer in this context window
        start_positions.append(cls_index)
        end_positions.append(cls_index)

        if len(answer["answer_start"]) == 0:
            continue

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        # Identify the context-token boundaries
        context_positions = [
            j for j, sid in enumerate(sequence_ids)
            if sid == 1
        ]

        if not context_positions:
            continue

        token_start = context_positions[0]
        token_end = context_positions[-1]

        # The complete answer must fit in this window
        if (
            offsets[token_start][0] > start_char
            or offsets[token_end][1] < end_char
        ):
            continue

        while (
            token_start <= token_end
            and offsets[token_start][0] <= start_char
        ):
            token_start += 1

        while (
            token_end >= token_start
            and offsets[token_end][1] >= end_char
        ):
            token_end -= 1

        start_positions[-1] = token_start - 1
        end_positions[-1] = token_end + 1

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized


train_features = train_data.map(
    preprocess_qa,
    batched=True,
    remove_columns=train_data.column_names,
    desc="Preprocessing training data"
)

validation_features = validation_data.map(
    preprocess_qa,
    batched=True,
    remove_columns=validation_data.column_names,
    desc="Preprocessing validation data"
)

print("Training features:", len(train_features))
print("Validation features:", len(validation_features))
print("Columns:", train_features.column_names)
print("Preprocessing completed!")


Preprocessing training data:   0%|          | 0/10000 [00:00<?, ? examples/s]

Preprocessing validation data:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training features: 10124
Validation features: 1019
Columns: ['input_ids', 'attention_mask', 'start_positions', 'end_positions']
Preprocessing completed!


In [ ]:

import torch
from transformers import AutoModelForQuestionAnswering

MODEL_NAME = "FacebookAI/roberta-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME
).to(device)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForQuestionAnswering LOAD REPORT from: FacebookAI/roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
qa_outputs.weight         | MISSING    | 
qa_outputs.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda
GPU: Tesla T4
Trainable parameters: 124056578


In [ ]:

from transformers import TrainingArguments, Trainer, DefaultDataCollator

training_args = TrainingArguments(
    output_dir="/content/qa_model",
    eval_strategy="steps",
    eval_steps=250,
    save_steps=250,
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    report_to="none",
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_features,
    eval_dataset=validation_features,
    data_collator=DefaultDataCollator()
)

trainer.train()


Step,Training Loss,Validation Loss
250,1.945869,1.639660
500,1.717783,1.490035
750,1.662454,1.581274
1000,1.385377,1.515115
1250,1.432705,1.492693
1500,1.024628,1.486640
1750,1.032021,1.279243
2000,1.099320,1.293354
2250,1.025803,1.289078
2500,0.941136,1.313357


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2532, training_loss=1.44471240081305, metrics={'train_runtime': 1090.669, 'train_samples_per_second': 18.565, 'train_steps_per_second': 2.322, 'total_flos': 3968052547792896.0, 'train_loss': 1.44471240081305, 'epoch': 2.0})

In [ ]:
SAVE_PATH = "/content/roberta_qa_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/roberta_qa_model


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

SAVE_PATH = "/content/roberta_qa_model"

tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model = AutoModelForQuestionAnswering.from_pretrained(SAVE_PATH)
model.eval()
model.to("cuda")

context = """
Amazon EC2 provides resizable compute capacity in the cloud.
An Internet Gateway enables communication between resources
in a VPC and the internet when routing and security rules allow it.
"""

question = "What enables communication between a VPC and the internet?"

inputs = tokenizer(
    question,
    context,
    return_tensors="pt",
    truncation=True,
    max_length=384
).to("cuda")

with torch.no_grad():
    outputs = model(**inputs)

start = outputs.start_logits.argmax(dim=-1).item()
end = outputs.end_logits.argmax(dim=-1).item()

if end >= start and end - start < 30:
    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end + 1],
        skip_special_tokens=True
    )
else:
    answer = "No reliable answer found."

print("Question:", question)
print("Answer:", answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Question: What enables communication between a VPC and the internet?
Answer: An Internet Gateway


In [ ]:
# building the chatbot

!pip -q install gradio pymupdf


In [ ]:

import fitz
import re
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def answer_question(pdf_file, question):
    if pdf_file is None:
        return "Please upload a PDF first.", "", ""

    if not question or not question.strip():
        return "Please enter a question.", "", ""

    path = pdf_file if isinstance(pdf_file, str) else pdf_file.name

    try:
        pages = []
        with fitz.open(path) as doc:
            for page_no, page in enumerate(doc, 1):
                text = re.sub(r"\s+", " ", page.get_text()).strip()
                if text:
                    pages.append((page_no, text))
    except Exception as e:
        return f"Could not read PDF: {e}", "", ""

    if not pages:
        return "No readable text found in the PDF.", "", ""

    # Split text into overlapping passages
    passages = []
    for page_no, text in pages:
        words = text.split()
        for i in range(0, len(words), 120):
            chunk = " ".join(words[i:i + 180])
            if len(chunk.split()) >= 5:
                passages.append((page_no, chunk))

    question = question.strip()

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english", ngram_range=(1, 2)
        )
        texts = [p[1] for p in passages]
        matrix = vectorizer.fit_transform(texts + [question])
        similarities = cosine_similarity(
            matrix[-1], matrix[:-1]
        ).ravel()
    except ValueError:
        return "I couldn't find relevant information in this document.", "", ""

    ranked = similarities.argsort()[::-1]

    if similarities[ranked[0]] < 0.08:
        return (
            "I couldn't find this topic in the uploaded document.",
            "",
            ""
        )

    # Extract complete, relevant sentences
    query_terms = set(
        re.findall(r"\b[a-zA-Z]{3,}\b", question.lower())
    )
    stop_words = {
        "what", "which", "when", "where", "who", "why",
        "how", "the", "and", "for", "are", "does", "did",
        "explain", "describe", "define", "tell", "about",
        "with", "from", "give", "please", "briefly"
    }
    query_terms -= stop_words

    candidates = []
    seen = set()

    for idx in ranked[:8]:
        page_no, context = passages[idx]
        sentences = re.split(r"(?<=[.!?])\s+", context)

        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence.split()) < 6:
                continue

            normalized = re.sub(r"\W+", "", sentence.lower())
            if normalized in seen:
                continue

            sentence_terms = set(
                re.findall(r"\b[a-zA-Z]{3,}\b", sentence.lower())
            )
            overlap = len(query_terms & sentence_terms)

            if overlap == 0:
                continue

            score = (
                0.65 * float(similarities[idx])
                + 0.35 * overlap / max(len(query_terms), 1)
            )

            candidates.append((score, page_no, sentence))
            seen.add(normalized)

    if not candidates:
        return (
            "I couldn't find a sufficiently relevant answer in this document.",
            "",
            ""
        )

    candidates.sort(key=lambda x: x[0], reverse=True)

    # Keep the best relevant sentences
    brief_sentences = []
    brief_words = 0

    for _, _, sentence in candidates:
        count = len(sentence.split())
        if brief_words + count > 55:
            continue

        brief_sentences.append(sentence)
        brief_words += count

        if len(brief_sentences) >= 2:
            break

    if not brief_sentences:
        brief_sentences = [candidates[0][2]]

    brief = " ".join(brief_sentences)

    # Key points must be distinct from the brief answer
    brief_normalized = {
        re.sub(r"\W+", "", s.lower()) for s in brief_sentences
    }

    key_points = []
    for _, _, sentence in candidates:
        normalized = re.sub(r"\W+", "", sentence.lower())

        if normalized in brief_normalized:
            continue

        key_points.append("• " + sentence)
        brief_normalized.add(normalized)

        if len(key_points) == 4:
            break

    # Pages supporting the displayed answer
    displayed = brief_sentences + [
        p[2] for p in candidates
        if any(p[2] == k[2:] for k in key_points)
    ]

    source_pages = sorted({
        page_no for _, page_no, sentence in candidates
        if sentence in displayed
    })

    source = "Source page(s): " + ", ".join(map(str, source_pages))

    return (
        brief,
        "\n".join(key_points) if key_points else
        "No additional distinct key points found.",
        source
    )


In [ ]:

demo = gr.Interface(
    fn=answer_question,
    inputs=[
        gr.File(
            label="Upload Technical PDF",
            file_types=[".pdf"],
            type="filepath"
        ),
        gr.Textbox(
            label="Ask a Question",
            placeholder="What is the Waterfall Model?",
            lines=2
        )
    ],
    outputs=[
        gr.Textbox(label="Brief Answer", lines=4),
        gr.Textbox(label="Key Points", lines=7),
        gr.Textbox(label="Source", lines=2)
    ],
    title="Closed-Domain Technical Documentation QA Bot",
    description="Upload your university technical manual and ask questions."
)

demo.launch(share=False, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>